# Colab pipeline for SciML_CFD_Engine
This notebook sets up the environment, launches TensorBoard, lets you override key hyperparameters, runs training, and visualizes a pipe cross-section from the latest checkpoint.

Replace `YOUR_REPO_URL` below with your GitHub repository URL before running the first cell.

In [ ]:
# 1) Clone repository (replace YOUR_REPO_URL) and install requirements
!git clone YOUR_REPO_URL repo || true
%cd repo || true
# Install standard requirements; in Colab you may want the CUDA-enabled wheel for torch
!pip install -r cfd_engine/requirements.txt -q || true
# If torch not installed or GPU desired, uncomment and run the appropriate install command below.
# For Colab GPU (example - adjust CUDA version if needed):
# !pip install --upgrade pip
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

import torch
print("Torch version:", torch.__version__)
{
	"cells": [
		{
			"cell_type": "markdown",
			"metadata": { "language": "markdown" },
			"source": [
				"# Colab pipeline for SciML_CFD_Engine\n",
				"This notebook sets up the environment, launches TensorBoard, runs a parameter grid search over pump and spatial-weight settings, and visualizes results. Replace `YOUR_REPO_URL` before running."
			]
		},
		{
			"cell_type": "code",
			"metadata": { "language": "python" },
			"source": [
				"# Clone repository (replace YOUR_REPO_URL) and install basic requirements\n",
				"!git clone YOUR_REPO_URL repo || true\n",
				"%cd repo || true\n",
				"!pip install -r cfd_engine/requirements.txt -q || true\n",
				"import torch\n",
				"print('Torch version:', torch.__version__)\n",
				"print('CUDA available:', torch.cuda.is_available())\n"
			]
		},
		{
			"cell_type": "code",
			"metadata": { "language": "python" },
			"source": [
				"# Start TensorBoard for monitoring\n",
				"%load_ext tensorboard\n",
				"import os\n",
				"os.environ.setdefault('LOGDIR', '/content/logs')\n",
				"%tensorboard --logdir $LOGDIR --host 0.0.0.0 --port 6006"
			]
		},
		{
			"cell_type": "markdown",
			"metadata": { "language": "markdown" },
			"source": [
				"## Grid search: PUMP_FORCE_MAX x SPATIAL_WEIGHT_SLOPE\n",
				"This cell will iterate over pump strengths and spatial slope values, set `RUN_ID` for each run, and execute `main.py`. Each run will save checkpoints with the run id suffix.\n",
				"**Warning**: each run will execute training sequentially. Ensure you have enough runtime quota (GPU/TPU) before launching."
			]
		},
		{
			"cell_type": "code",
			"metadata": { "language": "python" },
			"source": [
				"import os, time\n",
				"pump_values = [0.01, 0.05, 0.1]\n",
				"slope_values = [0.5, 1.0, 2.0]\n",
				"# Epochs per run (user requested ~1000)",
				"os.environ['ADAM_EPOCHS'] = '1000'\n",
				"os.environ['LBFGS_EPOCHS'] = '0'\n",
				"os.environ['BATCH_INTERIOR'] = '2000'\n",
				"os.environ['BATCH_BOUNDARY'] = '400'\n",
				"os.environ['NTK_REG_WEIGHT'] = '1e-4'\n",
				"os.environ['NTK_CHECK_INTERVAL'] = '1000'\n",
				"os.environ['LAMBDA_POS'] = '10.0'\n",
				"os.environ['LAMBDA_PIN'] = '1.0'\n",
				"os.environ['LAMBDA_BC'] = '20.0'\n",
				"os.environ['INLET_VELOCITY'] = '1.0'\n",
				"os.environ['RADIUS'] = '0.5'\n",
				"os.environ['LENGTH'] = '3.0'\n",
				"for p in pump_values:\n",
				"  for s in slope_values:\n",
				"    run_id = f'pump{p}_s{s}_{int(time.time())}'\n",
				"    print('Starting run', run_id)\n",
				"    os.environ['RUN_ID'] = run_id\n",
				"    os.environ['PUMP_FORCE_MAX'] = str(p)\n",
				"    os.environ['SPATIAL_WEIGHT_SLOPE'] = str(s)\n",
				"    # ensure unique logdir per run\n",
				"    os.environ['LOGDIR'] = f'/content/logs/{run_id}'\n",
				"    # Run training (blocking):\n",
				"    ret = os.system('python cfd_engine/main.py')\n",
				"    print('Run finished', run_id, 'exit', ret)\n",
				"    # small pause between runs\n",
				"    time.sleep(2)\n"
			]
		},
		{
			"cell_type": "markdown",
			"metadata": { "language": "markdown" },
			"source": [
				"## Post-processing\n",
				"Run the following cell after the grid-search completes to visualize the best/last checkpoint."
			]
		},
		{
			"cell_type": "code",
			"metadata": { "language": "python" },
			"source": [
				"import os, glob, numpy as np, torch, matplotlib.pyplot as plt\n",
				"from src.models.networks import PINN3DEngine\n",
				"# find latest checkpoint\n",
				"ckpts = sorted(glob.glob('checkpoints/*pump*.pth'))\n",
				"if not ckpts:\n",
				"  print('No pump-run checkpoints found in checkpoints/')\n",
				"else:\n",
				"  latest = ckpts[-1]\n",
				"  print('Loading', latest)\n",
				"  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
				"  model = PINN3DEngine(hidden_dim=256, num_layers=6).to(device)\n",
				"  model.load_state_dict(torch.load(latest, map_location=device))\n",
				"  model.eval()\n",
				"  # visualize cross-section at x=1.0\n",
				"  slice_x = 1.0\n",
				"  radius = float(os.environ.get('RADIUS', '0.5'))\n",
				"  rs = np.linspace(0, radius, 100)\n",
				"  thetas = np.linspace(0, 2*np.pi, 200)\n",
				"  R_grid, T_grid = np.meshgrid(rs, thetas, indexing='xy')\n",
				"  Y = (R_grid * np.cos(T_grid)).ravel()\n",
				"  Z = (R_grid * np.sin(T_grid)).ravel()\n",
				"  X = np.full_like(Y, slice_x)\n",
				"  coords = np.stack([X, Y, Z], axis=1).astype(np.float32)\n",
				"  with torch.no_grad():\n",
				"    pts = torch.from_numpy(coords).to(device)\n",
				"    preds = model(pts).cpu().numpy()\n",
				"  U = preds[:,0].reshape(R_grid.shape)\n",
				"  plt.figure(figsize=(6,5))\n",
				"  plt.pcolormesh(R_grid * np.cos(T_grid), R_grid * np.sin(T_grid), U, shading='auto')\n",
				"  plt.colorbar(label='u (streamwise)')\n",
				"  plt.title(f'Cross-section u at x={slice_x} - {latest}')\n",
				"  plt.gca().set_aspect('equal')\n",
				"  plt.show()\n"
			]
		}
	],
	"metadata": {
		"kernelspec": {"name": "python3", "display_name": "Python 3"},
		"language_info": {"name": "python", "version": "3.x"}
	},
	"nbformat": 4,
	"nbformat_minor": 5
}